# School Calendar, Time Windows and Analysis Panels
## From imputed hourly series to the study panels

This notebook uses both imputed hourly series from notebook 00 (hybrid and MIMA) and the official SEP calendar to build all analysis inputs: time-window aggregates, the arrival-window day–zone panel and the panel of within-day gaps.

The windows and their rationale are:

- **Arrival window (07:00–09:00):** the school-arrival period. Mexican basic-education morning shifts concentrate student arrival around 7:30–8:00, motivating this two-hour window.
- **Comparison window (10:00–12:00):** mid-morning of the SAME day. It is intended to follow arrival traffic while sharing the day's season, weather setting and city-wide events. Its proximity makes it an internal pseudo-control, not an exact control: weather, traffic and chemistry can evolve during the morning.
- Each window averages two hourly records: labels 07 and 08 for arrival, and 10 and 11 for comparison. The interval notation assumes start-of-hour timestamps; the provider's timestamp convention remains to be confirmed from metadata.

**Data access.** The SIMA measurements used here are not
redistributed in this repository (SIMA provided them for this study on the condition
that they are not disseminated); the repository holds the code, the official school
calendar and all derived results. This notebook reads files that notebooks 00 and 01
build from those measurements, and its configuration cell stops with a clear message
if they are missing. To rebuild them, request the annual workbooks from SIMA, place
them in `data_raw/` and run notebook 00 (its header gives the exact file format).

**Role in the analysis.** Data preparation: the time windows, the day–zone panels and the within-day gaps that every later notebook uses. The analytical hierarchy of the whole repository is stated in the README.

## 1. Initial setup

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Plot style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['font.size'] = 10

contaminantes = ['CO', 'NO2', 'O3', 'PM10', 'PM2.5']

output_dir = Path('../data_processed')
graficos_dir = output_dir / 'calendario'
graficos_dir.mkdir(exist_ok=True)

print('Libraries imported successfully')

# Check required inputs (restricted data; see heading)
# Input-data check (restricted data; see heading)
archivos_requeridos = [output_dir / 'datos_horarios_imputados_hibrido.csv.gz',
                       output_dir / 'datos_horarios_imputados_MIMA.csv.gz',
                       output_dir / 'calendario_oficial_sep.csv']
faltantes = [str(a) for a in archivos_requeridos if not a.exists()]
if faltantes:
    raise FileNotFoundError(
        ''
        ''
        "Files derived from the SIMA measurements are missing; they are not redistributed in this repository. "
        "Request the annual workbooks from SIMA, place them in data_raw/ and run notebooks 00 and 01. "
        f"Missing: {faltantes}")
print(f"Input data found: {len(archivos_requeridos)} files")

Libraries imported successfully
Input data found: 3 files


## 2. Loading inputs and building time-window aggregates

Day–zone–window aggregates are built here directly from the imputed hourly series produced by notebook 00.

In [2]:
calendario = pd.read_csv('../data_processed/calendario_oficial_sep.csv', parse_dates=['date'])

horas_franja = {'pico_7_9': [7, 8], 'fuera_pico_10_12': [10, 11]}

def construir_agregados(ruta_horaria, nombre_dataset):
    '\n    Average by day, zone, window and pollutant from the imputed hourly series.\n    '
    horario = pd.read_csv(ruta_horaria, parse_dates=['date'])
    horario['fecha'] = horario['date'].dt.normalize()
    horario['hora'] = horario['date'].dt.hour
    piezas = []
    for franja, horas in horas_franja.items():
        sub_f = horario[horario['hora'].isin(horas)]
        agregado = (sub_f.groupby(['fecha', 'estacion', 'parametro'])['valor'].mean()
                    .unstack('parametro').reset_index()
                    .rename(columns={'fecha': 'date'}))
        agregado['franja_horaria'] = franja
        piezas.append(agregado)
    salida = pd.concat(piezas, ignore_index=True)
    print(f"[{nombre_dataset}] aggregates: {len(salida):,} rows "
          f"({salida['date'].nunique()} days x {salida['estacion'].nunique()} zones x 2 windows)")
    return salida

df_hibrido = construir_agregados('../data_processed/datos_horarios_imputados_hibrido.csv.gz', 'hibrido')
df_mima = construir_agregados('../data_processed/datos_horarios_imputados_MIMA.csv.gz', 'MIMA')

print("=" * 70)
print('INPUTS READY')
print("=" * 70)
print(f"Official calendar: {len(calendario)} days | school-term dates: {int(calendario['lectivo'].sum())} | "
      f"attendance-proxy days: {int(calendario['dia_clases'].sum())}")
print(f"Monitoring zones: {sorted(df_hibrido['estacion'].unique())}")

[hibrido] aggregates: 13,152 rows (1096 days x 6 zones x 2 windows)


[MIMA] aggregates: 13,152 rows (1096 days x 6 zones x 2 windows)
INPUTS READY
Official calendar: 1096 days | school-term dates: 884 | attendance-proxy days: 607
Monitoring zones: ['CENTRO', 'NORESTE3', 'NOROESTE3', 'NORTE2', 'SUR', 'SUROESTE2']


## 3. Calendar cutoffs for time-series analyses

The main cutoffs are documented STUDENT RETURN dates from notebook 00. Reference cutoffs for falsification are the midpoints of official long vacations, when no school-return jump is expected. The 2025–26 winter vacation starts near the end of the study window and lacks a post-cutoff period, so it is not included as a cutoff.

In [3]:
inicios_alumnos = ['2023-01-09', '2023-08-28', '2024-01-08', '2024-08-26', '2025-01-09',
                   '2025-09-01']

vacaciones_largas = [('2023-07-27', '2023-08-27'),   # Summer 2023
                     ('2023-12-18', '2024-01-05'),   # Winter 2023-24
                     ('2024-03-25', '2024-04-05'),   # Easter 2024
                     ('2024-07-17', '2024-08-25'),   # Summer 2024
                     ('2024-12-19', '2025-01-08'),   # Winter 2024-25
                     ('2025-07-17', '2025-08-31')]   # Summer 2025
cortes_placebo = []
for inicio, fin in vacaciones_largas:
    medio = pd.Timestamp(inicio) + (pd.Timestamp(fin) - pd.Timestamp(inicio)) / 2
    cortes_placebo.append(str(medio.normalize().date()))

pd.DataFrame({'inicio_clases': pd.to_datetime(inicios_alumnos)}).to_csv(
    output_dir / 'inicios_bloques_clase.csv', index=False)
pd.DataFrame({'corte_placebo': pd.to_datetime(cortes_placebo)}).to_csv(
    output_dir / 'cortes_placebo.csv', index=False)

print("=" * 70)
print('CUTOFFS DEFINED')
print("=" * 70)
print(f"Starts (students return): {inicios_alumnos}")
print(f"Reference cutoffs (vacation midpoints): {cortes_placebo}")
print(f"Saved to: {output_dir / 'inicios_bloques_clase.csv'} y {output_dir / 'cortes_placebo.csv'}")

CUTOFFS DEFINED
Starts (students return): ['2023-01-09', '2023-08-28', '2024-01-08', '2024-08-26', '2025-01-09', '2025-09-01']
Reference cutoffs (vacation midpoints): ['2023-08-11', '2023-12-27', '2024-03-30', '2024-08-05', '2024-12-29', '2025-08-08']
Saved to: ../data_processed/inicios_bloques_clase.csv y ../data_processed/cortes_placebo.csv


In [4]:
fig, ax = plt.subplots(figsize=(14, 3.5))

serie_lectivo = calendario.set_index('date')['lectivo']
ax.fill_between(serie_lectivo.index, 0, serie_lectivo.values, step='post',
                color='#3498db', alpha=0.6, label='School term (official SEP)')
for inicio in inicios_alumnos:
    t = pd.Timestamp(inicio)
    ax.axvline(t, color='#e74c3c', linestyle='--', linewidth=1.5)
    ax.text(t, 1.05, inicio, rotation=45, fontsize=8, ha='left', color='#e74c3c', fontweight='bold')

ax.set_ylim(0, 1.35)
ax.set_yticks([0, 1])
ax.set_yticklabels(['Vacation', 'School term'])
ax.set_xlabel('Date', fontsize=12, fontweight='bold')
ax.set_title('Official school calendar (SEP, basic education) with student return dates',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(graficos_dir / '02_calendario_oficial_timeline.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Plot saved to: {graficos_dir / '02_calendario_oficial_timeline.png'}")

Plot saved to: ../data_processed/calendario/02_calendario_oficial_timeline.png


## 4. Building the arrival-window analysis panels

Panel choices include pollutant-specific floors before logarithms, an official date-based school-term indicator, calendar variables including a four-level weekday/weekend-by-regime category, Fourier terms for annual seasonality, the PC1 index and ISO week as the error-clustering unit.

In [5]:
pisos_fisicos = {'CO': 0.05, 'NO2': 0.05, 'O3': 0.5, 'PM10': 1.0, 'PM2.5': 0.5}
mapa_temporada4 = {12: 'invierno', 1: 'invierno', 2: 'invierno',
                   3: 'primavera', 4: 'primavera', 5: 'primavera',
                   6: 'verano', 7: 'verano', 8: 'verano',
                   9: 'otono', 10: 'otono', 11: 'otono'}
mapa_lectivo = calendario.set_index('date')['lectivo']
mapa_dia_clases = calendario.set_index('date')['dia_clases']
mapa_tipo_dia = calendario.set_index('date')['tipo_dia']

def construir_panel_pico(df_agregado, nombre_dataset):
    '\n    Build the day-zone arrival-window panel with the official school-term indicator.\n    '
    df = df_agregado.copy()
    for c in contaminantes:
        df[c] = df[c].clip(lower=pisos_fisicos[c])

    panel = df[df['franja_horaria'] == 'pico_7_9'].copy()
    panel['clase'] = panel['date'].map(mapa_lectivo)
    panel['dia_clases'] = panel['date'].map(mapa_dia_clases)
    panel['tipo_dia'] = panel['date'].map(mapa_tipo_dia)
    panel = panel.dropna(subset=contaminantes + ['clase'])
    panel['clase'] = panel['clase'].astype(int)

    panel['dow'] = panel['date'].dt.dayofweek
    panel['mes'] = panel['date'].dt.month
    panel['anio'] = panel['date'].dt.year
    panel['finde'] = (panel['dow'] >= 5).astype(int)
    panel['temporada_inv'] = panel['mes'].isin([10, 11, 12, 1, 2, 3]).astype(int)
    panel['temporada4'] = panel['mes'].map(mapa_temporada4)

    t0 = panel['date'].min()
    panel['t'] = (panel['date'] - t0).dt.days
    dia_del_anio = panel['date'].dt.dayofyear
    for k in (1, 2, 3):
        panel[f'sin{k}'] = np.sin(2 * np.pi * k * dia_del_anio / 365.25)
        panel[f'cos{k}'] = np.cos(2 * np.pi * k * dia_del_anio / 365.25)

    for c in contaminantes:
        panel[f'l_{c}'] = np.log(panel[c])
    Z = (panel[contaminantes] - panel[contaminantes].mean()) / panel[contaminantes].std(ddof=0)
    U, S, Vt = np.linalg.svd(Z.values - Z.values.mean(0), full_matrices=False)
    pesos_pc1 = Vt[0]
    if pesos_pc1[contaminantes.index('PM10')] < 0:
        pesos_pc1 = -pesos_pc1
    panel['PC1'] = Z.values @ pesos_pc1
    var_explicada = (S**2 / (S**2).sum())[0]

    iso = panel['date'].dt.isocalendar()
    panel['wk'] = iso.year.astype(str) + '-' + iso.week.astype(str).str.zfill(2)

    print(f"[{nombre_dataset}] n = {len(panel)} | school-term dates in panel = "
          f"{int(panel.groupby('date')['clase'].first().sum())} | "
          f"PC1 varianza explicada = {var_explicada*100:.1f}%")
    return panel

panel_hibrido = construir_panel_pico(df_hibrido, 'hibrido')
panel_mima = construir_panel_pico(df_mima, 'MIMA')

panel_hibrido.to_csv(output_dir / 'panel_pico_hibrido.csv', index=False)
panel_mima.to_csv(output_dir / 'panel_pico_MIMA.csv', index=False)
print(f"\nHybrid panel saved to: {output_dir / 'panel_pico_hibrido.csv'}")
print(f"MIMA panel saved to:    {output_dir / 'panel_pico_MIMA.csv'}")

[hibrido] n = 6576 | school-term dates in panel = 884 | PC1 varianza explicada = 50.5%
[MIMA] n = 6576 | school-term dates in panel = 884 | PC1 varianza explicada = 50.6%



Hybrid panel saved to: ../data_processed/panel_pico_hibrido.csv
MIMA panel saved to:    ../data_processed/panel_pico_MIMA.csv


## 5. Within-day gap panel: arrival versus comparison window

### The mathematics of the gap: the first difference

For monitoring unit $s$ and date $t$, let $\bar y^{7\text{-}9}_{st}$ be the arrival-window mean and $\bar y^{10\text{-}12}_{st}$ the comparison-window mean:

$$ \text{gap}_{st} = \log \bar y^{7\text{-}9}_{st} - \log \bar y^{10\text{-}12}_{st}. $$

Subtracting logarithms corresponds to dividing levels: the gap is the log of the arrival/comparison RATIO on the same day. Any multiplicative factor shared by both windows on that day and at that monitoring unit appears in both terms and cancels. This is why regressing the gap on the school indicator in notebook 05 is the collapsed form of a stacked two-window panel with date-by-monitoring-unit fixed effects. In the stacked form, notebook 05's zone, month and weekday controls enter as interactions with the window:

$$ \log y_{stf} = \mu_{st} + \beta\,(\text{School}_t \times \text{Arrival}_f) + \cdots + \varepsilon_{stf}. $$

The date–unit intercept $\mu_{st}$ absorbs shared factors. The first difference, arrival minus comparison, is taken within a date–unit cell; $\beta$ compares those gaps across school-term and vacation dates. Factors that evolve differently between windows do not necessarily cancel.

In [6]:
def construir_gaps(df_agregado, nombre_dataset):
    '\n    gap = log(7-9 window) - log(10-12 window), same day and zone, with official term indicator.\n    '
    df = df_agregado.copy()
    for c in contaminantes:
        df[c] = df[c].clip(lower=pisos_fisicos[c])

    pivote = df.pivot_table(index=['date', 'estacion'], columns='franja_horaria', values=contaminantes)
    gaps = pd.DataFrame(index=pivote.index)
    for c in contaminantes:
        gaps[f'gap_{c}'] = np.log(pivote[(c, 'pico_7_9')]) - np.log(pivote[(c, 'fuera_pico_10_12')])
    gaps = gaps.dropna().reset_index()

    gaps['clase'] = gaps['date'].map(mapa_lectivo).astype(int)
    gaps['dia_clases'] = gaps['date'].map(mapa_dia_clases)
    gaps['tipo_dia'] = gaps['date'].map(mapa_tipo_dia)
    gaps['dow'] = gaps['date'].dt.dayofweek
    gaps['mes'] = gaps['date'].dt.month
    gaps['anio'] = gaps['date'].dt.year
    gaps['finde'] = (gaps['dow'] >= 5).astype(int)
    gaps['temporada_inv'] = gaps['mes'].isin([10, 11, 12, 1, 2, 3]).astype(int)
    gaps['temporada4'] = gaps['mes'].map(mapa_temporada4)
    iso = gaps['date'].dt.isocalendar()
    gaps['wk'] = iso.year.astype(str) + '-' + iso.week.astype(str).str.zfill(2)
    gaps['wknum'] = iso.year.astype(int) * 100 + iso.week.astype(int)

    print(f"[{nombre_dataset}] gap panel: n = {len(gaps)}")
    return gaps

gaps_hibrido = construir_gaps(df_hibrido, 'hibrido')
gaps_mima = construir_gaps(df_mima, 'MIMA')

gaps_hibrido.to_csv(output_dir / 'gaps_franjas_hibrido.csv', index=False)
gaps_mima.to_csv(output_dir / 'gaps_franjas_MIMA.csv', index=False)
print(f"\nHybrid gaps saved to: {output_dir / 'gaps_franjas_hibrido.csv'}")
print(f"MIMA gaps saved to:    {output_dir / 'gaps_franjas_MIMA.csv'}")

[hibrido] gap panel: n = 6576
[MIMA] gap panel: n = 6576



Hybrid gaps saved to: ../data_processed/gaps_franjas_hibrido.csv
MIMA gaps saved to:    ../data_processed/gaps_franjas_MIMA.csv


## 6. Counts by the four-level category: weekday/weekend × school term/vacation

In [7]:
conteo_tipo_dia = (panel_hibrido.groupby('tipo_dia')['date'].nunique()
                   .reindex(['habil_lectivo', 'habil_vacaciones', 'finde_lectivo', 'finde_vacaciones']))
print("=" * 70)
print('PANEL DAYS BY DAY TYPE (weekday/weekend x school term/vacation)')
print("=" * 70)
print(conteo_tipo_dia.to_string())
print('\nThe weekday comparison is habil_lectivo vs habil_vacaciones;')
print('finde_lectivo vs finde_vacaciones is the weekend reference contrast (no school arrival).')

print('\nCOMPLETED')

PANEL DAYS BY DAY TYPE (weekday/weekend x school term/vacation)
tipo_dia
habil_lectivo       628
habil_vacaciones    155
finde_lectivo       256
finde_vacaciones     57

The weekday comparison is habil_lectivo vs habil_vacaciones;
finde_lectivo vs finde_vacaciones is the weekend reference contrast (no school arrival).

COMPLETED
